In [ ]:
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.tools import tool
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph , START, END


In [ ]:
llm = ChatOllama(model = "mistral:latest")
embeddings = OllamaEmbeddings(model = "mistral:latest")

In [ ]:
loader = PyPDFLoader("./dl-curriculum.pdf")
docs = loader.load()

In [ ]:
len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chuks = splitter.split_documents(docs)

In [ ]:
vectorstore = FAISS.from_documents(chuks, embedding=embeddings)


In [ ]:
vectorstore

In [ ]:
retriver = vectorstore.as_retriever(search_type= "similarity", kwargs={"k":3})

In [ ]:
prompt = "What is the main curriculum of the document?"

In [ ]:
retrived_docs = retriver._get_relevant_documents(prompt)

In [ ]:
@tool 
def rag_tool(query: str)-> str:
    """Retrive relevent information from the pdf document. 
    Use this tool when the use ask factual / conceptual questions
     that might be answered by the content of the pdf"""
    
    result = retriver.invoke(query)
    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query, 
        'context': context, 
        'metadata': metadata
    }


In [ ]:
tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def chat_node(state: ChatState):
    message = state['messages']
    response = llm_with_tools.invoke(message)
    return {"message": response}

In [ ]:
tool_node = ToolNode(tools=tools)

In [ ]:
graph = StateGraph()

graph.add_node("chat_node", chat_node)
graph.add_node("tools", tool_node)

graph.add_edge(START, "chat_node")
graph.add_conditional_edges("chat_node", tools_condition)
graph.add_edge('tools', "chat_node")

chatbot = graph.compile()


In [ ]:
chatbot

In [ ]:
result = chatbot.invoke({
    "messages": {
        HumanMessage(
            content= "Using the pdf note, explain how to find the ideal value of the k in the k means clustering problem"
        )
    }
 })

In [ ]:
print(result['messages'][-1])